# MCP Client Quickstart: Calling the Ethical AI Server

**Learning Objectives:**
- Understand how an MCP *client* connects to an MCP *server*
- Call tools, read resources, and use prompts programmatically
- See the difference between local (stdio) and remote (HTTP) transport

**Prerequisites:**
- Completed `05-mcp-security-audit.ipynb` or familiar with MCP concepts
- Python 3.11+, `uv` installed
- `server.py` from this directory

**Time Required:** ~20 minutes

---

## How MCP Works (Quick Recap)

```
┌────────────────────┐        stdio or HTTP        ┌────────────────────┐
│   MCP Client       │ ◄─────────────────────────► │   MCP Server       │
│  (this notebook)   │   JSON-RPC messages         │  (server.py)       │
└────────────────────┘                             └────────────────────┘
         │                                                  │
  calls tools,                                    exposes tools,
  reads resources,                                resources, prompts
  uses prompts                                    fetched from GitHub
```

The **transport** is just the pipe — the protocol is the same whether local or cloud.

## 1. Install Dependencies

In [ ]:
# The mcp package includes both server and client SDKs
!pip install -q mcp httpx

## 2. Connect Locally via stdio Transport

The stdio transport launches `server.py` as a subprocess and communicates over stdin/stdout.
This is the standard approach for local development.

> **Note:** Make sure you're running this notebook from the `08-mcp-server/` directory,
> or update the path to `server.py` below.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Point to server.py — adjust path if running from a different directory
SERVER_PATH = "server.py"

server_params = StdioServerParameters(
    command="uv",
    args=["run", SERVER_PATH],
)

async def list_server_capabilities():
    """Connect and print everything the server exposes."""
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools     = await session.list_tools()
            resources = await session.list_resources()
            prompts   = await session.list_prompts()

            print("=== TOOLS ===")
            for t in tools.tools:
                print(f"  {t.name}: {t.description}")

            print("\n=== RESOURCES ===")
            for r in resources.resources:
                print(f"  {r.uri}: {r.description}")

            print("\n=== PROMPTS ===")
            for p in prompts.prompts:
                print(f"  {p.name}: {p.description}")

await list_server_capabilities()

## 3. Call the `ping` Tool

`ping` is a fast diagnostic tool — no arguments, instant response.
Always call it first to confirm the connection is healthy.

In [ ]:
async def call_ping():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ping", {})
            print(result.content[0].text)

await call_ping()

## 4. Call the `search_guidelines` Tool

This tool searches all markdown documents in the repo for a keyword.
On the first call the server fetches all files from GitHub (~25 files).
Subsequent calls within 5 minutes use the cache.

In [ ]:
async def search(query: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("search_guidelines", {"query": query})
            print(result.content[0].text[:2000])  # truncate for readability

# Try any keyword from the repo — synthetic data, HIPAA, poisoning, etc.
await search("synthetic data")

## 5. Call the `get_learning_path` Tool

Returns only the section of `LEARNING_PATHS.md` that matches the role keyword.
Try: `Beginner`, `Healthcare`, `Security`, `Compliance`, `Advanced`.

In [ ]:
async def get_path(role: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("get_learning_path", {"role": role})
            print(result.content[0].text)

await get_path("Security")

## 6. Read a Resource

Resources are read-only data endpoints — think of them as documents the LLM can attach as context.
The URIs were listed in Section 2.

In [ ]:
async def read_resource(uri: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.read_resource(uri)
            # result.contents is a list; each item has a .text attribute
            text = result.contents[0].text
            print(text[:2000])  # truncate for readability

# Available URIs:
#   ethical-ai://governance/eu-ai-act
#   ethical-ai://healthcare/hipaa-checklist
#   ethical-ai://agentic-safety/mcp-threats
#   ethical-ai://tools/nemo-guardrails
await read_resource("ethical-ai://healthcare/hipaa-checklist")

## 7. Get a Prompt

Prompts are pre-built instruction templates that attach live context (e.g. the HIPAA checklist)
and return a ready-to-use system message for an LLM.

In Claude Desktop, prompts appear in the `/` slash command menu.
Here we retrieve one programmatically to see what it contains.

In [ ]:
async def get_prompt(name: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.get_prompt(name, {})
            for msg in result.messages:
                print(f"[{msg.role}]\n{msg.content.text[:1000]}...\n")

# Available prompts: audit_agent_security, review_healthcare_compliance
await get_prompt("review_healthcare_compliance")

## 8. Connect to the Remote Server (Streamable HTTP)

When deployed to Google Cloud Run, the server uses Streamable HTTP transport.
The client API is identical — only the connection setup changes.

> Set `CLOUD_RUN_URL` to your deployed Cloud Run service URL.
> The MCP endpoint is always at `/mcp`.

In [ ]:
from mcp.client.streamable_http import streamablehttp_client

CLOUD_RUN_URL = "https://your-service-name.run.app/mcp"  # replace with your URL

async def ping_remote():
    async with streamablehttp_client(CLOUD_RUN_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ping", {})
            print(result.content[0].text)

# Uncomment to test against Cloud Run:
# await ping_remote()

---

## Key Takeaways

| Concept | What You Saw |
|---|---|
| **Transport** | `stdio_client` (local) vs `streamablehttp_client` (remote) — same API |
| **Tools** | Called via `session.call_tool(name, args)` — returns structured content |
| **Resources** | Read via `session.read_resource(uri)` — read-only document endpoints |
| **Prompts** | Retrieved via `session.get_prompt(name, args)` — pre-built LLM instructions |
| **Cache** | First tool call fetches ~25 files from GitHub; subsequent calls are instant |

## Next Steps

- Explore `05-mcp-security-audit.ipynb` to learn how to audit MCP servers like this one
- Review `server.py` to see how each tool, resource, and prompt is implemented
- Read `05-agentic-safety/safe-mcp-patterns.md` for production security patterns
- Deploy to Cloud Run and test Section 8 against your live server

## Resources

- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk) — client and server SDK
- [FastMCP](https://gofastmcp.com) — the framework used in `server.py`
- [MCP Specification](https://modelcontextprotocol.io/specification) — protocol reference